# Build train / val / test files for the student (Mistral 7B)

- **train.jsonl**: teacher-labelled silver texts that passed validation
- **val.jsonl**: 50 pilot gold texts + 50 more gold texts (used during training to decide when to stop)
- **test.jsonl**: all remaining gold texts. The 150-text prompt pool is excluded, because it helped build the teacher's prompt.

The student gets **only the raw work order** and must produce the full answer: entity text + detailed type, and relations.
Relations point to entities by their position in the entity list (0, 1, 2...).

In [ ]:
import json, os, random, datetime
from collections import Counter

if not os.path.exists("/content/maintie"):
    os.system("git clone -q https://github.com/nlp-tlp/maintie.git /content/maintie")
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/mistral_ft/distill"
DATA_DIR = "/content/drive/MyDrive/mistral_ft/data"
os.makedirs(DATA_DIR, exist_ok=True)

gold = json.load(open("/content/maintie/data/gold_release.json"))
silver = json.load(open("/content/maintie/data/silver_release.json"))
labels = [json.loads(l) for l in open(f"{OUT_DIR}/silver_selfhosted_nothink.jsonl") if l.strip()]
PROMPT_POOL = set(json.load(open(f"{OUT_DIR}/prompt_pool_indices.json"))["prompt_pool_indices"])
PILOT = json.load(open(f"{OUT_DIR}/pilot_indices.json"))["pilot_indices"]
assert not set(PILOT) & PROMPT_POOL, "pilot overlaps prompt pool"
print(len(labels), "teacher labels |", len(gold), "gold |", len(PROMPT_POOL), "in prompt pool")

In [ ]:
INSTRUCTION = ("Extract the entities (with their MaintIE types) and the relations from this maintenance "
               "work order. Answer with JSON only.\nWork order: ")

def target(tokens, entities, relations):
    return json.dumps({
        "entities": [{"text": " ".join(tokens[e["start"]:e["end"]]), "type": e["type"]} for e in entities],
        "relations": [{"head": r["head"], "tail": r["tail"], "type": r["type"]} for r in relations],
    }, ensure_ascii=False)

def example(text, tgt):
    return {"messages": [{"role": "user", "content": INSTRUCTION + text},
                         {"role": "assistant", "content": tgt}]}

# ---- train: silver spans + teacher's detailed types and relations ----
train, train_ids, skipped = [], [], 0
for r in labels:
    if r["errors"] or r["pred"] is None:
        skipped += 1
        continue
    rec = silver[r["idx"]]
    types = {e["id"]: e["type"] for e in r["pred"]["entities"]}
    ents = [{"start": e["start"], "end": e["end"], "type": types[i]} for i, e in enumerate(rec["entities"])]
    train.append(example(rec["text"], target(rec["tokens"], ents, r["pred"].get("relations", []))))
    train_ids.append(r["idx"])

# ---- gold split: validation and test ----
rest = [i for i in range(len(gold)) if i not in PROMPT_POOL and i not in PILOT]
random.Random(2026).shuffle(rest)
VAL = sorted(PILOT + rest[:50])
TEST = sorted(rest[50:])
val = [example(gold[i]["text"], target(gold[i]["tokens"], gold[i]["entities"], gold[i]["relations"])) for i in VAL]
test = [example(gold[i]["text"], target(gold[i]["tokens"], gold[i]["entities"], gold[i]["relations"])) for i in TEST]

# Safety check: no gold test/val text may appear in training
train_texts = {silver[i]["text"] for i in train_ids}
overlap = sum(gold[i]["text"] in train_texts for i in VAL + TEST)
assert overlap == 0, f"{overlap} gold texts also appear in training data"

order = list(range(len(train)))
random.Random(42).shuffle(order)
for name, rows in [("train", [train[k] for k in order]), ("val", val), ("test", test)]:
    with open(f"{DATA_DIR}/{name}.jsonl", "w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

json.dump({
    "created": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "teacher": "Qwen/Qwen3.6-27B (vLLM, thinking off)",
    "label_file": "silver_selfhosted_nothink.jsonl",
    "train_silver_indices": [train_ids[k] for k in order],
    "val_gold_indices": VAL, "test_gold_indices": TEST,
    "excluded_prompt_pool": sorted(PROMPT_POOL),
}, open(f"{DATA_DIR}/splits.json", "w"))

print(f"train {len(train)} (skipped {skipped} failed labels) | val {len(val)} | test {len(test)}")
print("written to", DATA_DIR)

## Checks: type coverage and one example

In [ ]:
train_types = Counter(e["type"] for row in train for e in json.loads(row["messages"][1]["content"])["entities"])
test_types = Counter(e["type"] for i in TEST for e in gold[i]["entities"])
unseen = {t: c for t, c in test_types.items() if t not in train_types}
print(f"types in train: {len(train_types)} | types in test: {len(test_types)}")
print(f"test types never seen in training: {len(unseen)} ({sum(unseen.values())} test entities)")
print("most common unseen:", sorted(unseen.items(), key=lambda x: -x[1])[:10])
print("\nexample:\n", train[0]["messages"][0]["content"], "\n", train[0]["messages"][1]["content"])